# Sentri Project Documentation (AP CSP + Technical Design)

## 1) System Architecture & Data Flow
- [Idea Selection](https://github.com/Debuggers-CSP/Sentri-PRC/issues/1)
- [Problems to Fix in PRC](https://github.com/Debuggers-CSP/Sentri-PRC/issues/2)
- [Feature: SOS feature](https://github.com/Debuggers-CSP/Sentri-PRC/issues/3)
- [Feature: Sobriety Tracker](https://github.com/Debuggers-CSP/Sentri-PRC/issues/4)

### Architecture Overview
- **Frontend client (React + TypeScript)** renders pages, handles user interactions, and stores lightweight session/app data in browser local storage.
- **Auth context layer** centralizes session state (logged-in user, recommender completion, selected program updates).
- **Recovery tracker domain layer** computes streaks, gamification progress, and a rule-based support risk score from daily check-ins.
- **Persistence layer (browser storage)** stores user profile/session, recommender preferences, and per-program recovery check-ins.
 
### Data Flow Flowchart (System Architecture)
```mermaid
flowchart TD
A["User opens Sentri app"] --> B["UI Pages + Components"]
B --> C["AuthContext"]
B --> D["ProgramRecommender"]
B --> E["TrackerMain"]

C -->|login/logout/update profile| F[("localStorage: user")]
D -->|save selected tags| G[("localStorage: userPreferences")]

E --> H["useRecoveryData(program)"]
H --> I["Load state by storageKey"]
I --> J[("localStorage: sentri.recovery.program")]

E --> K["Check-in form submission"]
K --> L["submitCheckin()"]
L --> M["Append entry + XP/points + keep last 30"]
M --> J

H --> N["computeStreak + milestones"]
H --> O["evaluateSupportModel"]
N --> P["dashboardData"]
O --> P
P --> E
E --> Q["Garden/History/Rewards/Support UI"]
```
 
---
 
## 2) Technical Implementation Details
 
### Authentication & User Session
- Uses a React `AuthContext` provider for global user session state.
- `login(...)` stores normalized user fields (id, email, display name, first/last name, joined program) in both React state and local storage.
- `logout()` clears both in-memory state and persistent local storage to prevent accidental re-login on refresh.
- `updateJoinedProgram(programId)` immutably updates the active user record and persists it.
- `completeRecommender()` marks whether onboarding/recommender is completed.
 
### Data Storage / Schema
- **User object schema** includes:
- `id`, `name`, `username`, `email`
- `fname`, `lname`
- `joined_program`
- `hasCompletedRecommender`
- **Recovery check-in schema** includes:
- `date`
- `stayed_on_track_today`
- `mood_score`, `stress_score`, `craving_score`, `sleep_hours`
- `attended_meeting`, `exercise_done`
- `journal_note`
- `risk_level` (optional)
- `points_earned`
- **Recovery aggregate state** stores `points`, `xp`, and `checkins[]`.
- Program-scoped key format: `sentri.recovery.<program>`.
 
### AI Integration (Rule-Based Support Intelligence)
- The tracker includes an explainable scoring algorithm (`evaluateSupportModel`) that functions as a lightweight AI-style risk model.
- Inputs: current day behavioral and wellness factors from latest check-in (mood, stress, cravings, sleep, adherence).
- Weighted scoring adjusts risk up/down based on protective and risk factors (meeting attendance, exercise, sustained streak).
- Score is clamped to stable bounds, then mapped to `low \| medium \| high` risk bands.
- Output fields drive adaptive UX messaging:
- `ml_risk_level`
- `ml_risk_score`
- `ml_support_message`
- `ml_suggested_action`
 
### Program Recommender
- Preference tags are represented as a list of objects with stable IDs and labels.
- User selection is modeled as an array of selected IDs (`string[]`), toggled through inclusion/exclusion logic.
- On continue, selected preferences persist to local storage and user is navigated to program discovery.
 
---
 
## 3) AP CSP Concepts Applied
 
- **Program Purpose & Function**
- Sentri helps users track recovery behaviors, visualize progress, and receive support-oriented feedback.
 
- **Data Abstraction**
- Structured records (`User`, `CheckinEntry`, `RecoveryData`) reduce complexity and standardize how UI and algorithms access state.
 
- **Managing Complexity with Collections**
- Lists/arrays power preferences and check-in history.
- Mapping/filtering over arrays simplifies selection toggles, rendering, and history trimming.
 
- **Procedural Abstraction**
- Reusable procedures/functions such as `computeStreak`, `getNextMilestone`, `evaluateSupportModel`, and `submitCheckin` encapsulate logic and avoid duplication.
 
- **Algorithms (Sequencing, Selection, Iteration)**
- Sequencing: check-in processing pipeline (normalize input → compute score variables → update aggregate state).
- Selection: conditional thresholds to assign risk level and determine milestone states.
- Iteration: loops and list operations over check-ins and milestones.
 
- **Input / Output**
- Inputs: form entries (mood, stress, attendance, notes, tags).
- Outputs: visualized streaks/points/garden growth plus targeted support guidance.
 
- **Testing & Debugging Mindset**
- Deterministic, transparent risk scoring supports reproducible test cases.
- Local state persistence enables repeated scenario testing across sessions.
 
---
 
## 4) Code Highlights (Single Feature Focus)
 
### Feature: Daily Recovery Check-In + Adaptive Support Scoring
 
- **Procedure with parameters and algorithmic logic**
- `submitCheckin(formData)` accepts user input and normalizes values into a typed check-in record.
- `evaluateSupportModel(checkins, currentStreak)` computes weighted risk from latest data and returns support outputs.
 
- **Lists/dictionaries to manage complexity**
- `checkins[]` stores time-series behavior data.
- Each check-in acts like a dictionary/object with multiple health and behavior dimensions.
- Milestones are managed using a numeric array `[1, 3, 7, 14, 30, ...]`.
 
- **Input/output handling**
- Input from check-in forms is transformed with explicit type conversions (`Number`, `Boolean`, `String`) to reduce malformed state.
- Output populates dashboard cards, streak timeline, and support prompts.
 
- **Iteration and selection examples**
- Iteration computes streak by traversing sorted check-ins until first non-adherent day.
- Selection branches risk messages/actions based on score thresholds.
 
---
 
## 5) Testing Documentation
 
### Test Cases (Manual + Logic-Focused)
- **TC1: First-time user check-in initializes state**
- Input: empty storage, submit check-in with default values.
- Expected: points/xp increase by 10, one check-in saved, streak updates.
 
- **TC2: Streak increments across consecutive positive check-ins**
- Input: multiple sequential days with `stayed_on_track_today = true`.
- Expected: `computeStreak` returns growing streak, next milestone changes correctly.
 
- **TC3: Risk level escalation**
- Input: low mood, high stress/cravings, low sleep, `stayed_on_track_today = false`.
- Expected: model returns higher risk score and high-priority support message.
 
- **TC4: Protective behavior de-escalation**
- Input: attendance + exercise + multi-day streak.
- Expected: score reduction and lower risk label where thresholds are crossed.
 
- **TC5: Recommender selection toggle integrity**
- Input: click same preference repeatedly.
- Expected: item alternates between selected/unselected without duplicates.
 
- **TC6: Session persistence**
- Input: login and refresh.
- Expected: user restored from local storage; logout removes user state fully.
 
### Validation Strategy
- Validate numeric bounds and defaults for mood/stress/craving/sleep.
- Validate storage key isolation by recovery program.
- Validate `checkins` capped to latest 30 entries.
 
### Debugging Process
- Use deterministic scoring paths for risk model verification.
- Trace state updates in React hooks (`useEffect` load/save cycle).
- Use targeted console diagnostics around auth/session writes where needed.
- Reproduce edge cases by clearing local storage and replaying form inputs.